<a href="https://colab.research.google.com/github/totalleecher/ALAE/blob/master/Studio99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Studio99 on Google Colab & Kaggle (Full UI Parity + Multi-GPU VoxCPM2 + F5-Myanmar-TTS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bogyi2024/monk/blob/main/Studio99_Colab.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/Bogyi2024/monk/blob/main/Studio99_Colab.ipynb)

Run cells from top to bottom. Auto-detects **Google Colab** (T4/L4/A100) and **Kaggle** (**T4 x 2** parallel VoxCPM2).
- **Package Source:** Downloads `Studio99_Colab.zip` directly from `https://github.com/Bogyi2024/monk/raw/main/Studio99_Colab.zip` (with Google Drive / Kaggle Dataset fallback).
- **Google Colab:** Select **T4 GPU** in `Runtime -> Change runtime type`.
- **Kaggle:** Select **GPU T4 x2** + turn **Internet ON** in Settings.
- Keep the Cloudflare Quick Tunnel URL and one-time access code private.


In [ ]:
# [Step 1] Auto-detect Colab or Kaggle, download Studio99_Colab.zip from GitHub (Bogyi2024/monk), extract & configure dirs.
import os, glob, zipfile, tempfile, urllib.request

IS_KAGGLE = os.path.isdir('/kaggle/working')
WORK_ROOT = '/kaggle/working' if IS_KAGGLE else '/content'
STUDIO99_ROOT = os.path.join(WORK_ROOT, 'Studio99')
DATA_ROOT = '/kaggle/temp/Studio99Data' if IS_KAGGLE else '/content/Studio99Data'

# Direct GitHub raw download URL for Studio99_Colab.zip
DEFAULT_ZIP_URL = 'https://github.com/Bogyi2024/monk/raw/main/Studio99_Colab.zip'
STUDIO99_ZIP_URL = os.environ.get('STUDIO99_ZIP_URL', DEFAULT_ZIP_URL).strip()
if '/edit/' in STUDIO99_ZIP_URL or '/blob/' in STUDIO99_ZIP_URL:
    STUDIO99_ZIP_URL = STUDIO99_ZIP_URL.replace('/edit/', '/raw/').replace('/blob/', '/raw/')

def _download_zip(url, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    req = urllib.request.Request(url, headers={'User-Agent': 'Studio99-Colab/1.0'})
    with urllib.request.urlopen(req, timeout=120) as resp, open(dest, 'wb') as out:
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
    if not zipfile.is_zipfile(dest):
        os.remove(dest)
        raise ValueError(f'Downloaded file from {url} is not a valid ZIP archive (check repo visibility / URL).')

STUDIO99_ZIP = ''
if IS_KAGGLE:
    print('Platform detected: Kaggle')
    if STUDIO99_ZIP_URL:
        try:
            dl_path = '/kaggle/temp/Studio99_Colab.zip'
            print('Downloading Studio99_Colab.zip from:', STUDIO99_ZIP_URL)
            _download_zip(STUDIO99_ZIP_URL, dl_path)
            STUDIO99_ZIP = dl_path
        except Exception as e:
            print('GitHub download skipped/failed, checking Kaggle Dataset fallback:', e)
    if not STUDIO99_ZIP:
        candidates = [
            '/kaggle/working/Studio99_Colab.zip',
            *sorted(glob.glob('/kaggle/input/**/Studio99_Colab.zip', recursive=True)),
            *sorted(glob.glob('/kaggle/input/**/*Studio99*Colab*.zip', recursive=True)),
        ]
        STUDIO99_ZIP = next((p for p in candidates if os.path.isfile(p)), '')
    if not STUDIO99_ZIP:
        raise FileNotFoundError('Could not download Studio99_Colab.zip from GitHub or locate Kaggle Dataset.')
else:
    print('Platform detected: Google Colab')
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/Studio99Outputs', exist_ok=True)
    if STUDIO99_ZIP_URL:
        try:
            dl_path = '/content/Studio99_Colab.zip'
            print('Downloading Studio99_Colab.zip from:', STUDIO99_ZIP_URL)
            _download_zip(STUDIO99_ZIP_URL, dl_path)
            STUDIO99_ZIP = dl_path
        except Exception as e:
            print('GitHub download skipped/failed, falling back to Google Drive:', e)
    if not STUDIO99_ZIP:
        drive_zip = '/content/drive/MyDrive/Studio99_Colab.zip'
        if os.path.isfile(drive_zip):
            STUDIO99_ZIP = drive_zip
    if not STUDIO99_ZIP:
        raise FileNotFoundError('Could not download Studio99_Colab.zip from GitHub or find /content/drive/MyDrive/Studio99_Colab.zip')

os.makedirs(STUDIO99_ROOT, exist_ok=True)
with zipfile.ZipFile(STUDIO99_ZIP) as zf:
    base = os.path.realpath(STUDIO99_ROOT)
    for member in zf.infolist():
        dest = os.path.realpath(os.path.join(STUDIO99_ROOT, member.filename))
        if dest != base and not dest.startswith(base + os.sep):
            raise ValueError(f'Unsafe path in archive: {member.filename}')
    zf.extractall(STUDIO99_ROOT)
print('Studio99 source extracted:', STUDIO99_ROOT, 'from', STUDIO99_ZIP)

os.environ['STUDIO99_COLAB_DATA'] = DATA_ROOT
for name in ('tmp', 'cache', 'downloads', 'uploads', 'recap_jobs', 'outputs', 'split_outputs', 'dubbing_outputs', 'subtitle_outputs', 'storytelling_outputs', 'story_motion', 'logs', 'Studio99Desktop'):
    os.makedirs(os.path.join(DATA_ROOT, name), exist_ok=True)
os.environ['TMPDIR'] = os.environ['TMP'] = os.environ['TEMP'] = os.path.join(DATA_ROOT, 'tmp')
os.environ['XDG_CACHE_HOME'] = os.path.join(DATA_ROOT, 'cache')
os.environ['PIP_CACHE_DIR'] = os.path.join(DATA_ROOT, 'cache', 'pip')
os.environ['HF_HOME'] = os.path.join(DATA_ROOT, 'cache', 'huggingface')
os.environ['HF_HUB_CACHE'] = os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(DATA_ROOT, 'cache', 'huggingface', 'hub')
os.environ['TORCH_HOME'] = os.path.join(DATA_ROOT, 'cache', 'torch')
os.environ['PLAYWRIGHT_BROWSERS_PATH'] = os.path.join(DATA_ROOT, 'cache', 'ms-playwright')
os.environ['HF_HUB_DISABLE_XET'] = '1'
tempfile.tempdir = os.environ['TMPDIR']
print('Studio99 data root:', DATA_ROOT)


In [ ]:
# [Step 2] Install Linux + Python dependencies for ALL Studio99 tools (Colab & Kaggle)
!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-noto-core fonts-noto-extra

import os, sys, subprocess
os.chdir(STUDIO99_ROOT)
for rel_req in (
    os.path.join('recap_tool_src', 'requirements.txt'),
    os.path.join('tool_apps', 'dubbing', 'requirements.txt'),
    os.path.join('tool_apps', 'subtitle', 'requirements.txt'),
):
    req_path = os.path.join(STUDIO99_ROOT, rel_req)
    if os.path.isfile(req_path):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req_path])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'psutil', 'yt-dlp', 'edge-tts', 'gradio-client', 'google-genai', 'python-dotenv', 'Pillow', 'requests', 'voxcpm', 'soundfile', 'f5-myanmar-tts'])
subprocess.check_call([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'])
# CUDA 12 + cuDNN 9 runtime libs for faster-whisper / CTranslate2
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'])
import nvidia.cublas, nvidia.cudnn
cublas_lib = os.path.join(os.path.dirname(nvidia.cublas.__path__[0]), 'cublas', 'lib')
cudnn_lib = os.path.join(os.path.dirname(nvidia.cudnn.__path__[0]), 'cudnn', 'lib')
os.environ['LD_LIBRARY_PATH'] = ':'.join([cublas_lib, cudnn_lib, os.environ.get('LD_LIBRARY_PATH', '')])
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['PLAYWRIGHT_BROWSERS_PATH'] = os.path.join(os.environ['STUDIO99_COLAB_DATA'], 'cache', 'ms-playwright')
print('GPU:')
!nvidia-smi
subprocess.check_call([sys.executable, '-c', "import ctranslate2, torch; print('PyTorch CUDA GPUs:', torch.cuda.device_count() if torch.cuda.is_available() else 0); print('CTranslate2 CUDA device count:', ctranslate2.get_cuda_device_count())"])


In [ ]:
# [Step 2B - Optional] Install VoxCPM2-GGUF (Precompiled T4 sm_75 binary + Q8_0/F16 GGUF weights)
# Uses precompiled `voxcpm2-cli` from bogyi2024/voxcpm2colab (zero compile time) + DennisHuang648/VoxCPM2-GGUF.
import os, sys, tarfile, subprocess, urllib.request, glob

DATA_ROOT = os.environ.get('STUDIO99_COLAB_DATA', '/kaggle/temp/Studio99Data' if os.path.isdir('/kaggle/working') else '/content/Studio99Data')
GGUF_DIR = os.path.join(DATA_ROOT, 'models', 'voxcpm2_gguf')
os.makedirs(GGUF_DIR, exist_ok=True)

CLI_TAR_URL = os.environ.get(
    'STUDIO99_VOXCPM2_GGUF_TAR_URL',
    'https://huggingface.co/bogyi2024/voxcpm2colab/resolve/main/voxcpm2-cli-sm75.tar.gz',
).strip()
CLI_BIN = os.path.join(GGUF_DIR, 'voxcpm2-cli')

if not os.path.isfile(CLI_BIN):
    tar_path = os.path.join(GGUF_DIR, 'voxcpm2-cli-sm75.tar.gz')
    print('Downloading precompiled voxcpm2-cli (sm_75 / T4):', CLI_TAR_URL)
    req = urllib.request.Request(CLI_TAR_URL, headers={'User-Agent': 'Studio99-Colab/1.0'})
    with urllib.request.urlopen(req, timeout=180) as resp, open(tar_path, 'wb') as out:
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
    with tarfile.open(tar_path, 'r:gz') as tf:
        for m in tf.getmembers():
            name = os.path.basename(m.name)
            if name in ('voxcpm2-cli', 'BUILD_INFO.txt') and m.isfile():
                dest = os.path.join(GGUF_DIR, name)
                with tf.extractfile(m) as src, open(dest, 'wb') as dst:
                    dst.write(src.read())
    os.remove(tar_path)
os.chmod(CLI_BIN, 0o755)
build_info = os.path.join(GGUF_DIR, 'BUILD_INFO.txt')
if os.path.isfile(build_info):
    print('Binary build info:', open(build_info, encoding='utf-8').read().strip().replace('\n', ' | '))

# Ensure CUDA 13 / NCCL shared libs required by sm_75 binary are available without touching CUDA 12 libs
CU13_DIR = os.path.join(GGUF_DIR, 'cu13_libs')
def _add_cu13_paths():
    extras = [
        *sorted(glob.glob(os.path.join(CU13_DIR, 'nvidia', '*', 'lib'))),
        '/usr/local/cuda-13.0/lib64',
        '/usr/local/cuda-13/lib64',
        '/usr/local/cuda/lib64',
    ]
    cur = [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p]
    merged = []
    for p in extras + cur:
        if p and os.path.isdir(p) and p not in merged:
            merged.append(p)
    os.environ['LD_LIBRARY_PATH'] = ':'.join(merged)

_add_cu13_paths()
ldd_out = subprocess.run(['ldd', CLI_BIN], capture_output=True, text=True).stdout
missing_pkgs = []
if 'libcudart.so.13 => not found' in ldd_out:
    missing_pkgs.append('nvidia-cuda-runtime-cu13')
if 'libcublas.so.13 => not found' in ldd_out:
    missing_pkgs.append('nvidia-cublas-cu13')
if 'libnccl.so.2 => not found' in ldd_out:
    missing_pkgs.append('nvidia-nccl-cu13')
if missing_pkgs:
    print('Installing isolated CUDA 13 runtime libs for voxcpm2-cli:', ', '.join(missing_pkgs))
    os.makedirs(CU13_DIR, exist_ok=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--target', CU13_DIR, *missing_pkgs])
    _add_cu13_paths()

# Download VoxCPM2 GGUF weights (Q8_0 BaseLM ~1.73 GB recommended for T4 + Acoustic F16 ~1.7 GB)
from huggingface_hub import hf_hub_download
GGUF_REPO = os.environ.get('STUDIO99_VOXCPM2_GGUF_REPO', 'DennisHuang648/VoxCPM2-GGUF').strip()
BASE_FILE = os.environ.get('STUDIO99_VOXCPM2_GGUF_BASE_FILE', 'VoxCPM2-BaseLM-Q8_0.gguf').strip()
ACOUSTIC_FILE = os.environ.get('STUDIO99_VOXCPM2_GGUF_ACOUSTIC_FILE', 'VoxCPM2-Acoustic-F16.gguf').strip()

print(f'Downloading GGUF weights from {GGUF_REPO}: {BASE_FILE}, {ACOUSTIC_FILE} ...')
base_path = hf_hub_download(repo_id=GGUF_REPO, filename=BASE_FILE, local_dir=GGUF_DIR)
acoustic_path = hf_hub_download(repo_id=GGUF_REPO, filename=ACOUSTIC_FILE, local_dir=GGUF_DIR)

os.environ['STUDIO99_VOXCPM2_GGUF_CLI'] = CLI_BIN
os.environ['STUDIO99_VOXCPM2_GGUF_BASE'] = base_path
os.environ['STUDIO99_VOXCPM2_GGUF_ACOUSTIC'] = acoustic_path

# Verify binary runs cleanly with zero missing shared libraries
chk = subprocess.run([CLI_BIN, '--help'], capture_output=True, text=True)
help_out = (chk.stdout or '') + (chk.stderr or '')
if chk.returncode != 0 and 'Usage:' not in help_out:
    print('ldd output:\n', subprocess.run(['ldd', CLI_BIN], capture_output=True, text=True).stdout)
    raise RuntimeError(f'voxcpm2-cli --help failed (code {chk.returncode}): {chk.stderr or chk.stdout}')
print('✅ VoxCPM2-GGUF ready for T4!')
print('  CLI     :', CLI_BIN)
print('  BaseLM  :', base_path)
print('  Acoustic:', acoustic_path)


In [ ]:
# [Step 3 - Optional] Start gemini-web2api-go (localhost:8083) + Cloudflare Tunnel BEFORE launching Studio99.
# Skip this cell if you only want to use an official Google AI Studio Gemini API key.
import os, sys, re, json, time, secrets, getpass, subprocess, urllib.request

WORK_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
WEB2API_DIR = os.path.join(WORK_ROOT, 'web2api')
os.makedirs(WEB2API_DIR, exist_ok=True)
BIN_PATH = os.path.join(WEB2API_DIR, 'gemini-web2api-go')

if not os.path.isfile(BIN_PATH):
    print('Downloading latest gemini-web2api-go (linux_amd64)...')
    req = urllib.request.Request(
        'https://api.github.com/repos/zexadev/gemini-web2api-go/releases/latest',
        headers={'User-Agent': 'Studio99-Colab'},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        rel = json.loads(resp.read().decode('utf-8'))
    asset_url = next(
        (a['browser_download_url'] for a in rel.get('assets', []) if a.get('name', '').endswith('linux_amd64')),
        None,
    )
    if not asset_url:
        raise RuntimeError('Could not find linux_amd64 asset in zexadev/gemini-web2api-go latest release.')
    urllib.request.urlretrieve(asset_url, BIN_PATH)
    os.chmod(BIN_PATH, 0o755)
    print('Downloaded:', rel.get('tag_name'))

cookie_str = getpass.getpass('Gemini Web Cookie (SID=...; SAPISID=...; __Secure-1PSID=...; __Secure-1PSIDTS=...) [Enter to skip / add in Admin UI later]: ').strip()
cookie_file = os.path.join(WEB2API_DIR, 'cookie.txt')
cmd = [BIN_PATH, '--port', '8083', '--db', os.path.join(WEB2API_DIR, 'gemini.db')]
if cookie_str:
    with open(cookie_file, 'w', encoding='utf-8') as f:
        f.write(cookie_str + '\n')
    os.chmod(cookie_file, 0o600)
    cmd.extend(['--cookie-file', cookie_file])
del cookie_str

web2api_key = os.environ.get('STUDIO99_WEB2API_KEY') or ('sk-gemini-' + secrets.token_hex(16))
admin_token = os.environ.get('STUDIO99_WEB2API_ADMIN_TOKEN') or secrets.token_urlsafe(12)
cmd.extend(['--admin-token', admin_token])

# Stop any previous instance on port 8083
subprocess.run(['pkill', '-f', 'gemini-web2api-go'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared.*8083'], check=False)
time.sleep(0.5)

env = os.environ.copy()
env['API_KEY'] = web2api_key
env['ADMIN_TOKEN'] = admin_token
env['HOST'] = '0.0.0.0'
w2a_log_path = os.path.join(WORK_ROOT, 'web2api.log')
w2a_log = open(w2a_log_path, 'w', encoding='utf-8')
w2a_proc = subprocess.Popen(cmd, cwd=WEB2API_DIR, env=env, stdout=w2a_log, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        req = urllib.request.Request('http://127.0.0.1:8083/v1/models', headers={'Authorization': f'Bearer {web2api_key}'})
        with urllib.request.urlopen(req, timeout=2) as r:
            models_json = json.loads(r.read().decode('utf-8'))
        break
    except Exception:
        time.sleep(1)
else:
    with open(w2a_log_path, 'r', encoding='utf-8', errors='ignore') as lf:
        print(lf.read())
    raise RuntimeError('gemini-web2api-go did not start; see log above')

os.environ['STUDIO99_WEB2API_URL'] = 'http://127.0.0.1:8083/v1'
os.environ['STUDIO99_WEB2API_KEY'] = web2api_key
os.environ['STUDIO99_WEB2API_ADMIN_TOKEN'] = admin_token
avail = [m.get('id') for m in models_json.get('data', []) if isinstance(m, dict) and m.get('id')]
if avail:
    os.environ['STUDIO99_WEB2API_MODELS'] = ','.join(avail)

# Start Cloudflare Quick Tunnel so Admin UI works reliably on both Colab & Kaggle
cf_bin = os.path.join(WORK_ROOT, 'cloudflared')
if not os.path.isfile(cf_bin):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cf_bin,
    )
    os.chmod(cf_bin, 0o755)

cf_log_path = os.path.join(WORK_ROOT, 'web2api_tunnel.log')
cf_log = open(cf_log_path, 'w', encoding='utf-8')
w2a_tunnel = subprocess.Popen(
    [cf_bin, 'tunnel', '--url', 'http://127.0.0.1:8083', '--no-autoupdate'],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(25):
    time.sleep(1)
    with open(cf_log_path, 'r', encoding='utf-8', errors='ignore') as f:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
        if m:
            public_url = m.group(0)
            break

print('gemini-web2api-go is running at http://127.0.0.1:8083/v1')
print('Available models:', ', '.join(avail) if avail else '(anonymous / default)')
print('Admin Token:', admin_token)
if public_url:
    print('Admin Panel (Cloudflare URL):', public_url + '/admin')
    print('API Base URL (External):', public_url + '/v1')
else:
    print('Cloudflare tunnel failed; inspect', cf_log_path)


In [ ]:
# [Step 4] Configure Studio99 runtime & API keys (auto-wires gemini-web2api-go if started above).
import os, json, getpass
os.environ['STUDIO99_LOCAL'] = '1'
config_path = os.path.join(STUDIO99_ROOT, 'config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump({'local_mode': True, 'last_email': 'local@studio99'}, f, indent=2)

w2a_url = os.environ.get('STUDIO99_WEB2API_URL', '').strip()
w2a_key = os.environ.get('STUDIO99_WEB2API_KEY', '').strip()

prompt = 'Gemini API key (Enter to skip if using Web2API above, or paste official AI Studio key for Recap video): '
gemini_key = getpass.getpass(prompt).strip()
if not gemini_key and not (w2a_url and w2a_key):
    raise ValueError('Provide either a Gemini API key or run the gemini-web2api-go cell above first.')

key_dir = os.path.join(os.environ['STUDIO99_COLAB_DATA'], 'Studio99Desktop', 'run_keys')
os.makedirs(key_dir, exist_ok=True)
key_file = os.path.join(key_dir, '.env')
lines = []
if gemini_key:
    lines.append('GEMINI_IMAGE_PROJECT_1_KEY=' + gemini_key)
elif w2a_url and w2a_key:
    lines.append('GEMINI_IMAGE_PROJECT_1_KEY=custom-local-key')
if w2a_url and w2a_key:
    w2a_models = os.environ.get('STUDIO99_WEB2API_MODELS', '').strip() or 'gemini-3.5-flash-lite,gemini-3.6-flash,gemini-4-hydrogen'
    w2a_list = [m.strip() for m in w2a_models.split(',') if m.strip()]
    first_m = next((m for m in ('gemini-3.5-flash-lite', 'gemini-3.6-flash') if m in w2a_list), w2a_list[0] if w2a_list else 'gemini-3.5-flash-lite')
    lines.extend([
        'STUDIO99_API_MODE=openai',
        'STUDIO99_API_BASE_URL=' + w2a_url,
        'STUDIO99_API_KEY=' + w2a_key,
        'STUDIO99_API_MODELS=' + w2a_models,
        'GEMINI_TEXT_MODEL=' + first_m,
    ])
with open(key_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines) + '\n')
os.chmod(key_file, 0o600)
del gemini_key
print('Configured Studio99 keys in runtime-only storage.')
if w2a_url:
    print('Auto-connected Custom API ->', w2a_url, '(OpenAI Compatible, default model:', first_m + ')')


In [ ]:
# [Step 5] Verify faster-whisper GPU & VoxCPM2 Auto-Optimize in a subprocess (keeps Colab RAM free).
import sys, os, subprocess
check_code = (
    "import sys, ctranslate2; "
    "from faster_whisper import WhisperModel; "
    "n = ctranslate2.get_cuda_device_count(); "
    "print('CTranslate2 CUDA GPUs:', n); "
    "(_w := WhisperModel('small.en', device='cuda', compute_type='float16'), print('faster-whisper GPU check passed.')) if n > 0 else print('CUDA unavailable; CPU/int8 fallback.'); "
    f"sys.path.insert(0, {os.path.join(STUDIO99_ROOT, 'app')!r}); "
    "import voxcpm2_local, local_tts_engines; print('VoxCPM2 Hardware Profile:', voxcpm2_local.detect_hardware()); print('VoxCPM2-GGUF Status:', local_tts_engines.gguf_status())"
)
subprocess.check_call([sys.executable, '-c', check_code])


In [ ]:
# [Step 6] Start the Studio99 HTTP server and temporary Cloudflare Quick Tunnel (Colab & Kaggle).
import subprocess, sys, os, time, urllib.request, re, pathlib, secrets
WORK_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.environ['STUDIO99_COLAB_ACCESS_CODE'] = secrets.token_urlsafe(18)
access_code = os.environ['STUDIO99_COLAB_ACCESS_CODE']
server_log_path = os.path.join(WORK_ROOT, 'studio99_server.log')
tunnel_log_path = os.path.join(WORK_ROOT, 'studio99_tunnel.log')
subprocess.run(['pkill', '-f', 'colab_server.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared.*8765'], check=False)
time.sleep(0.5)
server_log = open(server_log_path, 'w', encoding='utf-8')
server = subprocess.Popen([sys.executable, os.path.join(STUDIO99_ROOT, 'app', 'colab_server.py'), '--host', '127.0.0.1', '--port', '8765'], cwd=STUDIO99_ROOT, stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8765/health', timeout=2).read()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f'Studio99 web server did not start; inspect {server_log_path}')
cloudflared = os.path.join(WORK_ROOT, 'cloudflared')
if not os.path.isfile(cloudflared):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    os.chmod(cloudflared, 0o755)
tunnel_log = open(tunnel_log_path, 'w', encoding='utf-8')
tunnel = subprocess.Popen([cloudflared, 'tunnel', '--url', 'http://127.0.0.1:8765', '--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(1)
    try:
        text = pathlib.Path(tunnel_log_path).read_text(encoding='utf-8', errors='replace')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if match:
            public_url = match.group(0)
            break
    except Exception:
        pass
if not public_url:
    raise RuntimeError(f'Cloudflare Quick Tunnel URL not found; inspect {tunnel_log_path}')
print('Open Studio99:', public_url)
print('One-time access code:', access_code)
print('Keep both private. When finished: tunnel.terminate(); server.terminate()')


## Supported Tools in Colab
- **Recap** (`recap_web`): Full planner + render workers, script review, overlays, and YouTube Auto mode.
- **Splitter** (`splitter_web`): Smart-cut FFmpeg splitting with frame preview and drag overlays.
- **Dubbing** (`dubbing_web`): Full video dubbing (Burmese / English, Edge TTS & Gemini TTS, Whisper GPU).
- **Subtitle** (`subtitle_web`): Video -> Burmese SRT / burned-in subtitles, optional existing SRT upload, split parts, Name Memory.
- **Original -> Recap** (`storytelling_web`): Movie/episode queue -> Burmese/multilingual storytelling recap or script-only export.
- **Long to Short** (`long_to_short`): Extract short/reel clips with Whisper GPU and Burmese/English captions.
- **Story Motion** (`story_motion`): AI story planner, scene image upload/FAL generation, browser voice test, and video render.
- **Novel** (`novel`): 4-part Burmese novel generator + covers.
- **Movie Review** (`movie_review`): Burmese movie review post, thumbnail, and Reel (add TMDB Bearer Token in Profile -> Keys).
- **Movie Search** (`movie_search`): Search, download, and send directly to Recap.
- **Auto Posting**: Remains desktop-only (requires persistent logged-in Facebook/TikTok browser profiles on a local PC).
- **VoxCPM2 (48kHz Local GPU TTS):** Integrated into **Recap** (`voxcpm2:*` voices + Reference Audio upload) and **Dubbing** (`VoxCPM2` provider + Auto Male/Female split). Model `openbmb/VoxCPM2` auto-downloads to `/content/Studio99Data/cache/huggingface` on first use.
- **VoxCPM2-GGUF (T4 Fast · Low RAM):** Run **[Step 2B - Optional]** to install the precompiled T4 (`sm_75`) `voxcpm2-cli` binary + `VoxCPM2-BaseLM-Q8_0.gguf` / `VoxCPM2-Acoustic-F16.gguf` weights, then select `voxcpm2_gguf:*` voices in **Recap** or **Dubbing**.